In [1]:
from langchain.agents.structured_output import ToolStrategy
from langchain.agents.middleware import ToolCallLimitMiddleware, AgentMiddleware, ModelRequest, wrap_tool_call
from langchain.agents import create_agent

from typing import Annotated, Sequence, TypedDict,Literal, List, Dict, Tuple, Union
import functools
import os
import threading

from langchain_core.tools import tool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    ToolMessage,
    AIMessage,
)
from langchain_anthropic import ChatAnthropic
# from langchain_openai import AzureChatOpenAI
# from langchain_deepseek import ChatDeepSeek

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, create_react_agent
from pydantic import BaseModel, Field

from src.tools import *
from src.prompt import dft_agent_prompt,hpc_agent_prompt,supervisor_prompt, oer_agent_prompt
from src import var

members = ["OER_Agent"]

class myStep(BaseModel):
    """Step in the plan."""

    step: str = Field(description="Step to perform.")
    agent: str = Field(
        description=f"Agent to perform the step. Should be one of {members}."
    )

class Plan(BaseModel):
    """Plan to follow in future"""

    steps: List[myStep] = Field(
        description=f"""
        Steps to follow in future. Each step is a tuple of (step, agent). agent can only be chosen from {members}.
        """
        # description="different steps to follow, should be in sorted order"
        # description="""different steps to follow (first element of the Tuple), and the agent in charge for each step (second element of the Tuple),
        # should be in sorted order by the order of execution"""
    )
    

class Response(BaseModel):
    """End everything and response to the user."""

    response: str


class Act(BaseModel):
    """Action to perform."""

    action: Union[Plan, Response] = Field(
        description="Action to perform. If you need to further use tools to get the answer, use Plan."
        "If you want to end the conversation, use Response."
        # "DO NOT use response unless absolutly necessary."
    )
    
class wokerResponse(BaseModel):
    """Response from the worker agent."""

    answer: str = Field(
        description="a short summary of the answer to the question or task."
    )
    
    summary: str = Field(
        description="""what have you done + what did you note down? i.e. I did xxx, and got xxx. I did xxx, and found xxx ..... In the end, I answered xxx/finished xxx/failed xxx/... I have noted down xxx, xxx, and xxx on CANVAS"""
    )

class PlanExecute(TypedDict):
    inputs: str
    plan: List[myStep]
    past_steps: List[myStep]
    response: str
    next: str

class DisableParallelToolCallsMiddleware(AgentMiddleware):
    
    def wrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return handler(request)
    
    async def awrap_model_call(self, request, handler):
        request.model_settings["parallel_tool_calls"] = False
        return await handler(request)

@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Only handle errors that occur during tool execution due to invalid inputs
        # that pass schema validation but fail at runtime (e.g., invalid SQL syntax).
        # Do NOT handle:
        # - Network failures (use tool retry middleware instead)
        # - Incorrect tool implementation errors (should bubble up)
        # - Schema mismatch errors (already auto-handled by the framework)
        #
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

def print_stream(s):
    if "messages" not in s:
        print("#################")
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write("#################\n")
        print(s)
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write(repr(s))
                f.write("\n")
    else:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
            if var.my_SAVE_DIALOGUE:
                with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(repr(message))
                    f.write("\n")
        else:
            if hasattr(message, 'usage_metadata'):
                var.TOKEN_USAGE.append(message.usage_metadata)
                print(f"input_tokens: {message.usage_metadata['input_tokens']}, output_tokens: {message.usage_metadata['output_tokens']}")
                if var.my_SAVE_DIALOGUE:
                    with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                        f.write(f"input_tokens: {message.usage_metadata['input_tokens']}, output_tokens: {message.usage_metadata['output_tokens']}\n")
            message.pretty_print()
            if var.my_SAVE_DIALOGUE:
                with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(message.pretty_repr())
                    f.write("\n")
    print()
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write("\n")

/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/autocat/data/lattice_parameters/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
MACE imported successfully


In [2]:
def supervisor_chain_node(state, agent, name):
    
    # read "status.txt" in the working directory
    with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
        status = f.read()
    while status == "stop":
        print(f"Calculation pause, supervisor is waiting. cwd: {var.my_WORKING_DIRECTORY}")
        # wait for 5 second
        time.sleep(5)
        with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
            status = f.read()
    
    print(f"supervisor is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"supervisor is processing!!!!!\n")

    print(state)
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(str(state))
            f.write("\n")
            
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(plan))
    # task_formatted = f"""For the following plan:
    # {plan_str}\n\nYou are tasked with executing step {1}, {task}."""
    old_tasks_string = "\n".join(f"{i+1}. {step.agent}: {step.step}" for i, step in enumerate(state["past_steps"]))
    
    supervisorMessage =  f"""
Your available agents are: {members}.

The overall goal is: {state['inputs']}. 

the current plan is:
{plan_str}

this is what has been done:
{old_tasks_string}

Please update the plan accordingly.
    """
        
    for agent_response in agent.stream(
        {"messages": [("user", supervisorMessage)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    ):
        # set agent_response to be the value of the first key of the dictionary
        agent_response = next(iter(agent_response.values()))
        print_stream(agent_response)

    # output = agent.invoke(
    #     {"messages": [("user", supervisorMessage)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    #     )
    
    agent_response = agent_response['structured_response']
    if isinstance(agent_response.action, Response):
        return {"response": agent_response.action.response, "next": "FINISH"}
    # elif isinstance(output.action, Response):
    #     return {"response": "Plan is not finished! Do not use response!", "next": "Supervisor"}
    else:
        plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(agent_response.action.steps))
        print(plan_str)
        if var.my_SAVE_DIALOGUE:
            with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
                f.write(plan_str)
                f.write("\n")
        return {"plan": agent_response.action.steps, "next": agent_response.action.steps[0].agent}

In [3]:
def worker_agent_node(state, agent, name, past_steps_list):
    # read "status.txt" in the working directory
    with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
        status = f.read()
    while status == "stop":
        print(f"Calculation pause, {name} Agent is waiting. cwd: {var.my_WORKING_DIRECTORY}")
        # wait for 5 second
        time.sleep(5)
        with open(f"{var.my_WORKING_DIRECTORY}/status.txt", "r") as f:
            status = f.read()
    
    print(f"Agent {name} is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"Agent {name} is processing!!!!!\n")
        
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step.step}" for i, step in enumerate(plan))
    # print(plan_str)
    # if var.my_SAVE_DIALOGUE:
    #     with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
    #         f.write(plan_str)
    #         f.write("\n")
    task = plan[0]
#     task_formatted = f"""For the following plan:
# {plan_str}\n\nYou are tasked with executing step {1}, {task}."""
    old_tasks_string = "\n".join(f"{i+1}. {step.agent}: {step.step}" for i, step in enumerate(past_steps_list))
    task_formatted = f"""
Here are what has been done so far:
{old_tasks_string}

Here is the overall objective:
{state["inputs"]}

Now, you are tasked with: {task}. Please only do this task! Do not do anything else! Please note down important information on CANVAS before you end.
"""
    
    print(task_formatted)
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(task_formatted)
            f.write("\n")
    print(f"Agent {name} is processing!!!!!")
    if var.my_SAVE_DIALOGUE:
        with open(f"{var.my_WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"Agent {name} is processing!!!!!\n")
    
    
    for agent_response in agent.stream(
        {"messages": [("user", task_formatted)]},  {"configurable": {"thread_id": "1"}, "recursion_limit": 1000}
    ):
        # set agent_response to be the value of the first key of the dictionary
        agent_response = next(iter(agent_response.values()))
        print_stream(agent_response)
    
    # agent_response = agent.invoke(
    #     {"messages": [("user", task_formatted)]},  {"configurable": {"thread_id": "1"}}
    # )
    structured_response = agent_response['structured_response']
    
    
    # past_steps_list.append((task, agent_response["messages"][-1].content))
    past_steps_list.append(myStep(step=structured_response.summary, agent=name))
    
    print_stream(structured_response.summary)
    
    return {
        "past_steps": past_steps_list,
    }
    
def whos_next(state):
    return state["next"]

In [4]:
### Prompt content
supervisor_prompt = f"""
<Role>:
    You are a scientist supervisor tasked with managing a research project. You have a team of worker agents, each with different expertise and capabilities. Your job is to coordinate the efforts of your team to achieve the research objectives efficiently and effectively.
<Objective>:
    Given the following user request, discuss with your worker agents, then decide which of the member to act next, and do what
<Instructions>:
    0,  You will be given a list of available workers, the overall objective from the user, a plan consists of a list of high level steps to achieve the objective, and a list of past steps that have been done.
    1.  If the plan is empty, For the given objective, first discuss with your worker agents, see what they can do and what are their opinions, then come up with a simple, high level research plan to achieve the objective.
        The plan should not contain detailed steps, only high level objectives, or milestones, your worker agents knows how to do it in detail.
        You don't have to use all the members, nor all the capabilities of the members.
        The result of the final step should be the final answer, but feel free to update and change the plan as you see fit. 
        Make sure to specify each step strictly and quatifiably - do not skip steps. (think about how a research plan should look like)
        
        If the plan is not empty, update the plan based on the current state of the project (check only related information on CANVAS and listen to the updates from the workers. Do not read through the entire CANVAS). 
        Remember to keep all steps that haven't been done yet. Only add steps to the plan that still NEED to be done. Do not return previously done steps as part of the plan.
    2.  Given the conversation above, suggest who should act next.
<Requirements>:
    0.  You MUST discuss with your worker agents to get their expert opinion before making a plan.
    1.  When you want to discuss with your worker agents, you can simply creat a plan with questions or contents of your discussion.
    2.  When creating a action about discussion, directly ask the question, do not say anything else. The worker agent will read the question and give you the answer, then you can update your plan based on the answer.
        """

dataset_description = """
<fields>
  <field name="Composition" type="str">
    <description>chemical formula, based on the composition of the entire unit cell</description>
  </field>

  <field name="Materialid" type="str">
    <description>unique id for the material.</description>
  </field>

  <field name="Reduced Formula" type="str">
    <description>reduced (simplest) chemical formula of the unit cell.</description>
  </field>

  <field name="Elements" type="List[str]">
    <description>list of all unique chemical element symbols in the material.</description>
  </field>

  <field name="NSites" type="int">
    <description>total number of atoms in the unit cell.</description>
  </field>

  <field name="Crystal System" type="str">
    <description>crystal system (e.g., cubic, tetragonal, orthorhombic, etc.).</description>
  </field>

  <field name="Dimensionality Cheon" type="str or float">
    <description>
      dimensionality label computed by get_dimensionality_cheon in pymatgen, indicating how the largest bonded cluster extends in space.
      Possible values: "3D", "2D", "1D", "0D", or "intercalated ion/molecule"; may be NaN if undetermined.
    </description>
  </field>

  <field name="Bandgap" type="numpy.float64">
    <description>bandgap from PBE-level DFT. NaN if not available.</description>
  </field>

  <field name="Disorder Probability" type="numpy.float64">
    <description>
      predicted probability of crystallographic substitutional disorder (0 = 0%, 1 = 100%), from a machine learning model developed by Jakob et al.
    </description>
  </field>

  <field name="average_HHI_P" type="numpy.int64" domain="production">
    <description>
      average Herfindahl-Hirschman Index (HHI) for global production concentration across producing countries, averaged over all elements (atoms) in the material.
    </description>
  </field>

  <field name="average_HHI_P_excluding_OHCNPS" type="numpy.int64" domain="production">
    <description>
      average Herfindahl-Hirschman Index (HHI) for global production concentration across producing countries, averaged over all elements (atoms) in the material excluding O, H, C, N, P, and S.
    </description>
  </field>

  <field name="max_HHI_P" type="numpy.int64" domain="production">
    <description>
      the maximum Herfindahl-Hirschman Index (HHI) for global production concentration across producing countries, amongst all elements in the material.
    </description>
  </field>

  <field name="average_HHI_R" type="numpy.int64" domain="reserve">
    <description>
      average Herfindahl-Hirschman Index (HHI) for global reserve concentration across all countries, averaged over all elements (atoms) in the material.
    </description>
  </field>

  <field name="average_HHI_R_excluding_OHCNPS" type="numpy.int64" domain="reserve">
    <description>
      average Herfindahl-Hirschman Index (HHI) for global reserve concentration across all countries, excluding O, H, C, N, P, and S.
    </description>
  </field>

  <field name="max_HHI_R" type="numpy.int64" domain="reserve">
    <description>
      the maximum Herfindahl-Hirschman Index (HHI) for global reserve concentration across all countries, amongst all elements in the material.
    </description>
  </field>
</fields>
"""

# oer_agent_prompt = f"""
#             <Role>: 
#                 You are a very powerful and yet obedient assistant that setup OER screening tasks and working in a team. You do exactly what you are told to do.
#                 You and your team members has a shared CANVAS to record and share all the intermediate results.
#                 You and your team members has a shared structured EXPLOG to record the progress of the OER study and HPC jobs, and it will be updated automaticlly. 
#                 Please strickly follow the tasks given, do not do anything else.
#             <Objective>: 
#                 You are responsible for providing suggestion on candidates with provided tools. 
#                 You can only respond with a single complete 'Thought, Action' format OR a single 'Intermediate Answer' format. 
#                 Please strickly follow the tasks given, do not do anything else.
#             <Your Capability>: (Only do what you are told to do)
#                 inspect and read the CANVAS with suitable tools to see what's available.
#                 conduct initial screening on the dataset to form initial candidates with the right tool.
#                 read dataframes on the CANVAS with the right tool.
#                 find out potential facets with a maximum miller index.
#                 determine the terminations for a given facet.
#                 study OER on a given termination using MLIP or VASP.
#                 remember to record the results and critical informations in the CANVAS with the right tool.
#             <Requirements>: 
#                 0. Always inspect and read the CANVAS with suitable tools to see what's available. Do not note down following info on CANVAS (they will be automatically noted down in the EXPLOG): cadidate_id, reason_for_cadidate_selection, 
#                 1. Before making any decisions on what to do next, always check the experimentLog to check the progress of the a certain study, and/or the progress of HPC jobs.
#                 2. when creating filters for a intial rough filtering, here are some info about the df {dataset_description}:
#                 3. Conduct arxiv searches to find relevant literatures when necessary (i.e. when choosing which filter to apply and how to sort the dataframe; or to choose which system as the candidates to study; or choose which sites to put O or OH onto).
#                 4. Please strickly follow the tasks given, do not do anything else.                
#                 5. If error occur, only response with 'Job failed' + error message. Do not say anything else.
#                 6. DO NOT conduct any inferenece on the result or conduct any post-processing unless explicitly asked.
#                 7. Do not give further suggestions on what to do next.
#                 8. The final answer should be concise summary in a sentence. Do not repeat what you've noted on the CANVAS, just mention it's on the CANVAS.
#                 9. You don't have to use all the tools provided, only use the tools that are necessary.
#                 10. Do not report absolute path.
#                 11. When asked to pick a candidate, do not do further analysis.
#             """

oer_agent_prompt = f"""
            <Role>: 
                You are a very powerful and yet obedient assistant that conduct screening for best catalsys for OER application
            <Objective>: 
                Given a dataset, since you cannot run calculations for all system, all surfaces, and all sites, 
                you need to determine what candidates to run calculation on. 
                For each candidates, you need to determine which terminations to study.
                For the termination, you need to determine which sites to adsorb O or OH onto.
                In the end, your gooal is to find which system with which termination and on which sites has the lowest idealOverPotential.
                You need to determine what would be the optimal screening strategy to explore the dataset, that ensures you finding the best cadidate while saving time and compute.
            <Your Capability>: (Only do what you are told to do)
                You and your team shares a common EXPLOG, which is a structures place that automatically record:
                    1. material_ID of the candidates you studied/studying
                    2. reason or hypothesis behind the choice of each candidate
                    3. study progress and HPC job status of a candidate: job_type, slurmID, status, termination_index, site_index, VASP_dir, processNote
                    4. note about each candidate study
                You can get a summary and/or query the EXPLOG, get updated information, which helps you decide what to do next
                You and your team shares a common CANVAS, where you can inspect, read, note down and share important information that is NOT in the EXPLOG
                You can perform literature search on arXiv
                You can filter and sort the dataset and save the result in a seperate dataframe. Here is some infomation about the dataset in dataframe format: {dataset_description}
                You can view a certain protion of the dataframe
                You can submit different types of calculations about a candidate (bulk relax, surface (termination) relax, and adsorption relax)
                You have a tool that can help you determine which termination to choose
                You have a tool that shows you site information given a termination
            <Requirements>: 
                Before you do anything, first to check the EXPLOG, and then base on the progress information, decide what to do next.
                If you need some other information, you can always inspect and extract information from CANVAS
                Conduct arxiv searches to find relevant literatures when necessary (i.e. when choosing which filter to apply and how to sort the dataframe; or to choose which system as the candidates to study; or choose which sites to put O or OH onto).
                Please follow the tasks strickly, do not do anything else, only do what you were told to do
                If you encounter any difficulties in DFT calculation tell the supervisor to ask help from DFT agent
                You will be notified once something is noted down in EXPLOG
                Remember to always note down important information that is NOT in the EXPLOG onto the CANVAS
                The final answer should be concise summary in a sentence. Do not repeat what you've noted on the CANVAS, just mention it's on the CANVAS.
                You don't have to use all the tools provided, only use the tools that are necessary.
                Do not report absolute path.
                Please note down your capability on CANVAS after you was asked about it.
            """

dft_agent_prompt ="""
            <Role>: 
                You are a very powerful and yet obedient assistant that performs density functional theory calculations and working in a team. You do exactly what you are told to do.
                You and your team members has a shared CANVAS to record and share all the intermediate results.
                Please strickly follow the tasks given, do not do anything else.
            <Objective>: 
                You are responsible for generating the quantum espresso input file for the given material and parameter setting with provided tools. 
                You can only respond with what you have done. i.e. answered the question, generated the input file, etc. DO NOT respond with the content or the result itself. 
                Please strickly follow the tasks given, do not do anything else.
            <Your Capability>: (Only do what you are told to do)
                inspect and read the CANVAS with suitable tools to see what's available.
                create valid input structure for the system of interest with the right tool.
                Find the correct pseduopotential filename using the tool provided (do not report the absolute path).
                Generate the quantum espresso input file with proper ASE tool. Pay attention to calculation type and funtional choice.
                Always generate conventional cell with ibrav=0 and do not use celldm and angstrom at the same time.
                If the system involves hubbard U correction, specify starting magnetization in SYSTEM card and hubbard U parameters in HUBBARD card, and use the pre-defined hubbard correction tool.
                Save all the files in pwi format and into job list and report to supervisor to let HPC Agent to submit the job. 
                generate convergence test scripts with a tool.
                determine the most optimal settings based on the convergence test.
                calculate lattice constant and formation energy based on the DFT calculation.
                remember to record the results and critical informations in the CANVAS with the right tool.
            <Requirements>: 
                0. You MUST be conservative on what you can do. Tools you have are your only capabilities. You can try to use them to achieve some higher level goals, but do not do anything that is not covered by the tools you have.
                1. Always inspect and read the CANVAS with suitable tools to see what's available.
                2. QE input files should be in pwi format, and output file will have .pwo appended to the filename.
                3. Do not generate convergence test for all systems and all configurations.
                4. Please only generate one batch of convergence test for the most complicated system using the most complicated configuration.
                5. Please strickly follow the tasks given, do not do anything else. 
                6. If everything is good, only response with the tool message and a short summary of what has been done. If you think it's the final answer, prefix 'Intermediate Answer'. Do not say anything else.
                7. If error occur, only response with 'Job failed' + error message. Do not say anything else.
                8. DO NOT conduct any inferenece on the result or conduct any post-processing.
                9. Once you done generating scripts, report back to the supervisor and stop immediately.
                10. Do not give further suggestions on what to do next.
                11. The electron conv_thr should be 1e-6.
                12. Use the right smearing based on the material.
                13. The final answer must NOT repeat what you've noted on the CANVAS, just mention it's on the CANVAS.
                14. You don't have to use all the tools provided, only use the tools that are necessary.
                15. Do not report absolute path.
                16. when calculating formation energies, convergence test on DFT parameters should be done on one representitive system with both the adsorbate and the surface.
                17. If a job is having issue, i.e. didn't converge or not accurate enough, use the right tool to get suggestions on how to modify the input file to fix the issue.
            """

dft_reader_agent_prompt = """
You are a DFT expert who's good at giving suggestions on how to solve convergence issues. You will be given a filename. Read only that file and provide feedback base on that file only. Do not try to read any other files. 
The input file will end with .pwi, the output file will end with .pwi.pwo
If you were given a input file, try to figure out why the job didn't converge base on the input file.
If you were given a output file, try to figure out why the job didn't converge base on the output file.
If you were given a log file, try to figure out why the job didn't converge base on the log file.
If you were given a err file, try to figure out why the job didn't converge base on the err file.
DO NOT READ ANY OTHER FILES!!!
You don't have abilities to do anything else or fix anything.
Please strickly follow the tasks given, do not do anything else.
"""

calculater_prompt = "You are very powerful assistant that performs bulk modulus calculations on atomistic level, but don't know current events. \
            For each query vailidate that the chemical elements only contains Copper and Gold and otherwise cancel. \
            Get the structure from supplied function. Use Atomic positions in Angstroms. \
            If the composition is not pure gold or pure copper, use the supplied function to generate mixed metal structure.\
            Calculate bulk modulus of both single metal and mixed metal from the supplied function.\
            You should try identifying if either Cu or Au meets the desired bulk modulus, if not, \
            try changing the concentration of Cu and Au until reaches 10 trials or meets the user input bulk modulus requirement.\
            From each calculation, validate that the desired bulk modulus is strictly following user input bulk modulus, otherwise cancel.\
            Also, is user specified a acceptable error range, for each calculation if the resulting bulk modulus is within that range, stop immediately.\
            "

HPC_resources = """
Artemis by the Numbers

Node     #   CPU         GPU          RAM      Disk   $
-----------------------------------------------------------
H100     3   AMD 9654    4x H100 SXM  768 GB   1.9 TB  117,950
A100     2   AMD 7513    4x A100 SXM  512 GB   1.6 TB  58,597
Largemem 3   AMD 9654                 768 GB   1.9 TB  13,989
CPU      25  AMD 9654                 368 GB   1.9 TB  12,998

CPU Specifications
----------------------------------------------
CPU                 Cores  Threads  Base    Boost             L3 Cache
AMD Epyc 9654 CPU    96     192     2.6 GHz 3.55 GHz (All Core)  384 MB
AMD Epyc 7513 CPU    32      64     2.6 GHz 3.65 GHz (Max)       128 MB

*Nodes are partitioned by threads, not cores. Picking 1 or a multiple of 2 is advisable; see sbatch's --distribution flag.

GPU Specifications
-------------------------------------------------
GPU        VRAM  GPU Mem Bandwidth  FP64  FP64 TC  FP32 - TC  BF16 TC
A100 SXM   80 GB  2,039 GB/s         9.7   19.5    156        312
H100 SXM   80 GB  3.34 TB/s          34    67      989        1989

*FLOPs are listed in teraFLOPs (10¹² floating point operations per second). Tensor Cores (TC) are specialized for general matrix multiplications (GEMM).

Partitions
-----------------------------------------------------------
Partition        Nodes         Max Wall Time  Priority  Max Jobs  Max Nodes
venkvis-cpu      CPU           48 hrs
venkvis-largemem Large Mem     48 hrs
venkvis-a100     A100          8 hrs
venkvis-h100     H100          8 hrs
"""

QE_submission_example = """
export OMP_NUM_THREADS=1

spack load quantum-espresso@7.2

echo "Job started on `hostname` at `date`"

mpirun pw.x -i [input_script_name.pwi] > [input_script_name.pwi].pwo

echo " "
echo "Job Ended at `date`"
"""


hpc_agent_prompt = f"""
            <Role>: 
                You are a very powerful high performance computing expert that runs calculations on the supercomputer, but don't know current events.
                Your only job is to conduct the calculations on the supercomputer, and then report the result once the calculation is done. 
                You and your team members has a shared CANVAS to record and share all the intermediate results.
                Please strickly follow the tasks given, do not do anything else.
            <Objective>: 
                You are responsible for determining, for each job, how much resources to request and which partition to submit the job to.
                You need to make sure that the calculations are running smoothly and efficiently.
                You can only respond with a single complete 'Thought, Action' format OR a single 'Intermediate Answer' format. 
            <Instructions>: 
                1. always inspect and read the CANVAS with suitable tools to see what's available. i.e. you can find what jobs to run from the CANVAS with the right key.
                2. Use the right tool to read one quantum espresso input file from the working directory and, one job by one job, determinie how much resources to request, which partition to submit that job to, and what would be the submission scipt based on the resources info {HPC_resources}. Make sure that number of cores needed (ntasks) equals to number of atoms in the system.
                3. Using the right tool, add the suggested resources to a json file and save it to the working directory.
                4. repeat the process until all resource suggestions are created.
                5. Use appropriate tool to submit all the jobs in the job_list.json to the supercomputer based on the suggested resource. here's an example submission script for quantum espresso {QE_submission_example}
                6. Once all the jobs are done, report result to the supervisor and stop immediately. 
                7. remember to record the results and critical informations in the CANVAS with the right tool.
            <Requirements>:
                0. You MUST be conservative on what you can do. Tools you have are your only capabilities. You can try to use them to achieve some higher level goals, but do not do anything that is not covered by the tools you have.
                1. follow the instruction strictly, do not do anything else.
                2. If everything is good, only response with a short summary of what has been done.
                3. If error occur, only response with 'Job failed' + error message. Do not say anything else.
                4. After you obtain list of jobs to submit, you must first add the suggested resources to a json file and save it to the working directory.
                5. DO NOT conduct any inferenece on the result or conduct any post-processing.
                6. Do not give further suggestions on what to do next.
            """

meam_doc = """
.. index:: pair_style meam
.. index:: pair_style meam/kk
.. index:: pair_style meam/ms
.. index:: pair_style meam/ms/kk
pair_style meam command
=========================
Accelerator Variants: *meam/kk*
pair_style meam/ms command
==========================
Accelerator Variants: *meam/ms/kk*
Syntax
.. code-block:: LAMMPS
   pair_style style
* style = *meam* or *meam/ms*
Examples
.. code-block:: LAMMPS
   pair_style meam
   pair_coeff * * ../potentials/library.meam Si ../potentials/si.meam Si
   pair_coeff * * ../potentials/library.meam Ni Al NULL Ni Al Ni Ni
   pair_style meam/ms
   pair_coeff * * ../potentials/library.msmeam H Ga ../potentials/HGa.meam H Ga
Description
.. note::
   The behavior of the MEAM potential for alloy systems has changed
   as of November 2010; see description below of the mixture_ref_t
   parameter
Pair style *meam* computes non-bonded interactions for a variety of
materials using the modified embedded-atom method (MEAM) :ref:`(Baskes)
<Baskes>`.  Conceptually, it is an extension to the original :doc:`EAM
method <pair_eam>` which adds angular forces.  It is thus suitable for
modeling metals and alloys with fcc, bcc, hcp and diamond cubic
structures, as well as materials with covalent interactions like silicon
and carbon.
The *meam* pair style is a translation of the original Fortran version
to C++. It is functionally equivalent but more efficient and has
additional features. The Fortran version of the *meam* pair style has
been removed from LAMMPS after the 12 December 2018 release.
Pair style *meam/ms* uses the multi-state MEAM (MS-MEAM) method
according to :ref:`(Baskes2) <Baskes2>`, which is an extension to MEAM.
This pair style is mostly equivalent to *meam* and differs only
where noted in the documentation below.
In the MEAM formulation, the total energy E of a system of atoms is
given by:
.. math::
   E = \sum_i \left\{ F_i(\bar{\rho}_i)
       + \frac{1}{2} \sum_{i \neq j} \phi_{ij} (r_{ij}) \right\}
where *F* is the embedding energy which is a function of the atomic
electron density :math:`\rho`, and :math:`\phi` is a pair potential
interaction.  The pair interaction is summed over all neighbors J of
atom I within the cutoff distance.  As with EAM, the multi-body nature
of the MEAM potential is a result of the embedding energy term.  Details
of the computation of the embedding and pair energies, as implemented in
LAMMPS, are given in :ref:`(Gullet) <Gullet>` and references therein.
The various parameters in the MEAM formulas are listed in two files
which are specified by the :doc:`pair_coeff <pair_coeff>` command.
These are ASCII text files in a format consistent with other MD codes
that implement MEAM potentials, such as the serial DYNAMO code and
Warp.  Several MEAM potential files with parameters for different
materials are included in the "potentials" directory of the LAMMPS
distribution with a ".meam" suffix.  All of these are parameterized in
terms of LAMMPS :doc:`metal units <units>`.
Note that unlike for other potentials, cutoffs for MEAM potentials are
not set in the pair_style or pair_coeff command; they are specified in
the MEAM potential files themselves.
Only a single pair_coeff command is used with the *meam* style which
specifies two MEAM files and the element(s) to extract information
for.  The MEAM elements are mapped to LAMMPS atom types by specifying
N additional arguments after the second filename in the pair_coeff
command, where N is the number of LAMMPS atom types:
* MEAM library file
* Element1, Element2, ...
* MEAM parameter file
* N element names = mapping of MEAM elements to atom types
See the :doc:`pair_coeff <pair_coeff>` page for alternate ways
to specify the path for the potential files.
As an example, the ``potentials/library.meam`` file has generic MEAM
settings for a variety of elements.  The ``potentials/SiC.meam`` file
has specific parameter settings for a Si and C alloy system.  If your
LAMMPS simulation has 4 atoms types and you want the first 3 to be Si,
and the fourth to be C, you would use the following pair_coeff command:
.. code-block:: LAMMPS
   pair_coeff * * library.meam Si C sic.meam Si Si Si C
The first 2 arguments must be \* \* so as to span all LAMMPS atom types.
The first filename is the element library file. The list of elements following
it extracts lines from the library file and assigns numeric indices to these
elements. The second filename is the alloy parameter file, which refers to
elements using the numeric indices assigned before.
The arguments after the parameter file map LAMMPS atom types to elements, i.e.
LAMMPS atom types 1,2,3 to the MEAM Si element.  The final C argument maps
LAMMPS atom type 4 to the MEAM C element.
If the second filename is specified as NULL, no parameter file is read,
which simply means the generic parameters in the library file are
used.  Use of the NULL specification for the parameter file is
discouraged for systems with more than a single element type
(e.g. alloys), since the parameter file is expected to set element
interaction terms that are not captured by the information in the
library file.
If a mapping value is specified as NULL, the mapping is not performed.
This can be used when a *meam* potential is used as part of the
*hybrid* pair style.  The NULL values are placeholders for atom types
that will be used with other potentials.
.. note::
   If the second filename is NULL, the element names between the two
   filenames can appear in any order, e.g. "Si C" or "C Si" in the
   example above.  However, if the second filename is **not** NULL (as in the
   example above), it contains settings that are indexed **by numbers**
   for the elements that precede it.  Thus you need to ensure that you list
   the elements between the filenames in an order consistent with how the
   values in the second filename are indexed.  See details below on the
   syntax for settings in the second file.
"""

md_agent_prompt = f"""
            <Role>:
                You are a very powerful molecular dynamics expert that runs simulations on the supercomputer, but don't know current events.
            <Objective>:
                You are responsible for generating the LAMMPS input file for a givin simulation with provided tools. 
                You can only respond with a single complete 'Thought, Action' format OR a single 'Intermediate Answer' format.
            <Instructions>:
                1. find which potential to use for the simulation.
                2. Use the right tool to generate initial structure for the simulation
                3. Generate the input script.
                4. Save all the files in to job list and report to supervisor to let HPC Agent to submit the job.                 
            <Requirements>:
                1. Please follow the tasks strickly, do not do anything else. 
                2. If everything is good, only response with the tool message and a short summary of what has been done. If you think it's the final answer, prefix 'Intermediate Answer'. Do not say anything else.
                3. If error occur, only response with 'Job failed' + error message. Do not say anything else.
                4. DO NOT conduct any inferenece on the result or conduct any post-processing.
                5. Once you done generating scripts, report back to the supervisor and stop immediately.
                6. Do not give further suggestions on what to do next.
            """

In [5]:
def create_planning_graph(config: dict) -> StateGraph:
    # create a file named status.txt in the working directory
    WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
    with open(f"{WORKING_DIRECTORY}/status.txt", "w") as f:
        f.write("run")
    
    # Define the model
    # llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    # workerllm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0, tool_choice="auto")
    llm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    workerllm = ChatAnthropic(model="claude-sonnet-4-5-20250929", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0, tool_choice="auto")
    # workerllm = ChatAnthropic(model="claude-3-5-sonnet-20241022", api_key=config['ANTHROPIC_API_KEY'],temperature=0.0)
    # llm = AzureChatOpenAI(model="gpt-4o", api_version="2024-08-01-preview", api_key=config["OpenAI_API_KEY"], azure_endpoint = config["OpenAI_BASE_URL"])
    # workerllm = AzureChatOpenAI(model="gpt-4o", api_version="2024-08-01-preview", api_key=config["OpenAI_API_KEY"], azure_endpoint = config["OpenAI_BASE_URL"], model_kwargs={'parallel_tool_calls': False})
    # llm = ChatDeepSeek(model_name=config["DeepSeek_MDL"], api_key=config['DeepSeek_API_KEY'], api_base=config['DeepSeek_BASE_URL'], temperature=0.0)
    
    var.my_WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
    
    if not eval(config["SAVE_DIALOGUE"]):
        var.my_SAVE_DIALOGUE = False
    
    
    
    # System Supervisor with tool bind and with_structured_output
    
    # supervisor_chain = supervisor_prompt | llm.bind_tools(supervisor_tools).with_structured_output(Act)
    # supervisor_agent = functools.partial(supervisor_chain_node, chain=supervisor_chain, name="Supervisor")
    
    
        
    
    # def supervisor_agent(state):
    #     print("Supervisor!!!!!!!!!")
    #     supervisor_chain = (
    #         prompt
    #         | llm.with_structured_output(routeResponse)
    #     )
    #     return supervisor_chain.invoke(state)
    
    ## Memory Saver
    memory = MemorySaver()

    PAST_STEPS = []
    myCANVAS = {}
    
    supervisor_tools = [
        inspect_my_canvas,
        read_my_canvas,
        inspect_explog
        ]
    
    supervisor_agent = create_agent(
        model=llm,
        tools=supervisor_tools, 
        system_prompt=supervisor_prompt,
        # Structured output via ToolStrategy (tool-calling fallback)
        response_format=ToolStrategy(Act),  # Or ProviderStrategy for native models
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    
    # supervisor_agent = create_react_agent(llm, tools=supervisor_tools,
    #                                prompt=supervisor_prompt, response_format=Act)   
    supervisor_node = functools.partial(supervisor_chain_node, agent=supervisor_agent, name="Supervisor_Agent")
    
    ### DFT Agent
    dft_tools = [
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        calculate_formation_E,
        generateSurface_and_getPossibleSite,
        generate_myAdsorbate,
        add_myAdsorbate,
        init_structure_data,
        find_pseudopotential,
        write_QE_script_w_ASE,
        calculate_lc,
        generate_convergence_test,
        get_kspacing_ecutwfc,
        generate_eos_test,
        read_energy_from_output,
        get_convergence_suggestions,
        analyze_BEEF_result
        ]
    # dft_agent = create_react_agent(workerllm, tools=dft_tools,
    #                                prompt="You are a DFT expert")   
    dft_agent = create_agent(
        model=workerllm,
        tools=dft_tools,
        system_prompt=dft_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    dft_node = functools.partial(worker_agent_node, agent=dft_agent, name="DFT_Agent", past_steps_list=PAST_STEPS)

    
    oer_tools = [
        inspect_explog,
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        OER_data_analasis_v2,
        read_df,
        arXiv_search,
        enter_candidate_in_log,
        submit_dft_job,
        get_terminations_ranking,
        list_adsorption_sites,
        read_explog,
        get_top_k_candidates,
        extract_df
        ]
    # oer_agent = create_react_agent(workerllm, tools=oer_tools,
    #                                prompt=oer_agent_prompt)
    oer_agent = create_agent(
        model=workerllm,
        tools=oer_tools,
        system_prompt=oer_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    oer_node = functools.partial(worker_agent_node, agent=oer_agent, name="OER_Agent", past_steps_list=PAST_STEPS)

    ### HPC Agent
    # hpc_tools = [read_script, submit_and_monitor_job, read_energy_from_output]
    hpc_tools = [
        inspect_my_canvas,
        write_my_canvas,
        read_my_canvas,
        submit_and_monitor_job,
        add_resource_suggestion
        ]

    # hpc_agent = create_react_agent(workerllm, tools=hpc_tools,
    #                                prompt=hpc_agent_prompt)
    hpc_agent = create_agent(
        model=workerllm,
        tools=hpc_tools,
        system_prompt=hpc_agent_prompt,
        response_format=ToolStrategy(wokerResponse),
        middleware=[DisableParallelToolCallsMiddleware(), handle_tool_errors]
    )
    hpc_node = functools.partial(worker_agent_node, agent=hpc_agent, name="HPC_Agent", past_steps_list=PAST_STEPS)
    
    ### MD Agent
    # md_tools = [
    #     find_classical_potential,
    #     init_structure_data,
    #     write_LAMMPS_script
    # ]
    
    # md_agent = create_react_agent(llm, tools=md_tools,
    #                               state_modifier=md_agent_prompt)
    
    # md_node = functools.partial(worker_agent_node, agent=md_agent, name="MD_Agent", past_steps_list=PAST_STEPS)

    # save_graph_to_file(dft_agent, config['working_directory'], "dft_agent")
    


    # Create the graph
    graph = StateGraph(PlanExecute)
    # graph.add_node("DFT_Agent", dft_node)
    # graph.add_node("HPC_Agent", hpc_node)
    graph.add_node("OER_Agent", oer_node)
    # graph.add_node("MD_Agent", md_node)
    # graph.add_node("CSS_Agent", css_node)

    graph.add_node("Supervisor", supervisor_node)
    
    for member in members:
    # We want our workers to ALWAYS "report back" to the supervisor when done
        graph.add_edge(member, "Supervisor")
    # The supervisor populates the "next" field in the graph state
    # which routes to a node or finishes
    conditional_map = {k: k for k in members}
    conditional_map["FINISH"] = END
    conditional_map["Supervisor"] = "Supervisor" 
    graph.add_conditional_edges("Supervisor", whos_next, conditional_map)
    graph.add_edge(START, "Supervisor") 
    # return graph.compile(checkpointer=memory)
    return graph.compile()

In [6]:
userMessage_6 = "You are going to calculate the lattice constant for BCC Li through DFT, the experiment value is 3.451, use this to create the initial structure."
userMessage_7 = "You are going to generat a Pt surface structure with 2x2x4 supercell, then do a convergence test, use maximum ecutwfc = 160. Get the optimal kspacing and ecutwfc."
userMessage_8 = """Please generate intial structures required to calculate CO adsorbtion on Pt(111) surface with 1/4 coverage (2x2x4 supercell), and calculate the adsorbtion energy."""
userMessage_9 = """
Please find out the most perfered adsorbtion site and adsorbate orientation (up or down) for CO adsorbtion on Pt(111) surface with 1/4 coverage (2x2x4 supercell).
"""
userMessage_10 = """please find the adsorption energy difference between the most favorable configurations (different adsorbate orientations 0, 90, 180) at fcc site and
most favorable configuration (different adsorbate orientations 0, 90, 180) at ontop site for CO on Pt(111) surface with p(2x2) adsorbate overlayer (1/4 coverage). 
Please use PBE pseudopotential and PBE exchange correlation function.
Literatures suggest that ontop site is 0.108 eV less stable than fcc site when using PBE xc. 
If your result is not within 10 percent of the literature, please find out possible reasons and resolve it."""

userMessage_11 = "I am trying to study adsorption of CO on Pt111 surface at fcc site. Job CO_Pt111_fcc_upright_k_0.3_ecutwfc_60.pwi did not converge, please figure out why and resolve the convergence issue."

userMessage_12 = """please find the adsorption energy difference between the most favorable configurations (different adsorbate orientations 0, 90, 180) at fcc site and most favorable configuration (different adsorbate orientations 0, 90, 180) at ontop site for CO on Pt(111) surface with p(2x2) adsorbate overlayer (1/4 coverage), and analyze the uncertainty.
Please use PBE pseudopotential and Bayesian Error Estimation Functional (BEEF) exchange correlation function.
Literatures suggest that ontop site is 0.18 eV less stable than fcc site when using PBE xc.
If your result is not within 10 percent of the literature, please find out possible reasons and resolve it."""

userMessage_13 = """please conduct a initial screening on the default dataset on potential candidates as a catalyst for OER reaction. Please only consider O only and skip the study of OH and OOH for now. Please save the cadidates dataframe into a csv file."""

userMessage_14 = """please conduct a OER screening study to find out the best system to use as catalyst for OER reaction. You must evaluate more less than 3 different systems. Please only consider O only and skip the study of OH and OOH for now. Available systems can be found in the default dataset. you must include Ir, with Nsite < 20. Please use VASP as the calculator. 
"""

testMessage = """
please conduct a OER screening study to find out the best system to use as catalyst for OER reaction.
You must decide a screening strategy for selecting candidates materials and performing relavent DFT calculation to evaluate the OER activity.
Please only consider O only and skip the study of OH and OOH for now. 
Available systems can be found in the default dataset. you must use Nsite < 20. Please use VASP as the calculator. 
"""

config = load_config(os.path.join('./config', "default.yaml"))
# check_config(config)

WORKING_DIRECTORY = var.my_WORKING_DIRECTORY
# print N number of '#', where n = len("##  Working directory: " + WORKING_DIRECTORY + " ##")
print("#" * (len("##  Working directory: " + WORKING_DIRECTORY + " ##")))
print("##  Working directory: " + WORKING_DIRECTORY + " ##")
print("#" * (len("##  Working directory: " + WORKING_DIRECTORY + " ##")))

assert WORKING_DIRECTORY is not None, "Please set the WORKING_DIRECTORY var"

CANVAS.set_working_directory(WORKING_DIRECTORY)
# CANVAS.canvas["finished_job_list"] = ["CO_Pt111_fcc_upright_k_0.3_ecutwfc_60.pwi"]

# set environment variable
os.environ["OMP_NUM_THREADS"] = "1"

# check if working directory exists, if so delete it
if os.path.exists(WORKING_DIRECTORY):
    os.system(f"rm -rf {WORKING_DIRECTORY}")

os.makedirs(WORKING_DIRECTORY, exist_ok=False)

EXPLOG.init(Path(WORKING_DIRECTORY)/"TEMP_vasp_calcs", "test")
# check if resource_suggestions.db exist in the working directory
db_file = os.path.join(WORKING_DIRECTORY, 'resource_suggestions.db')
if os.path.exists(db_file):
    os.remove(db_file)
initialize_database(db_file)

graph = create_planning_graph(config)
llm_config = {"thread_id": "1", 'recursion_limit': 1000}

# print(graph)


# save_graph_to_file(graph, WORKING_DIRECTORY, "super_graph")
# exit()


# for s in graph.stream(
# {
#     "messages": [
#         HumanMessage(content=f"{userMessage_4}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

# for s in graph.stream(
# {
#     "messages": [
#         HumanMessage(content=f"{userMessage_2}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

# for s in graph.stream(
# {
#     "input": f"{userMessage_6}",
#     "plan": [],
#     "past_steps": []
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")
        
# for s in graph.stream(
# {
#     "input": [
#         HumanMessage(content=f"{userMessage_5}")
#     ]
# },llm_config):
#     if "__end__" not in s:
#         print(s)
#         print("----")

print("Start, check the log file for details")
log_filename = f"./log/agent_stream_{int(time.time())}.log"  # Add timestamp to filename
with open(log_filename, "a") as log_file:
    log_file.write(f"=== Session started at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    if eval(config["SAVE_DIALOGUE"]):
        with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"=== Session started at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    
    for s in graph.stream(
        {
            "inputs": f"{testMessage}",
            "plan": [],
            "past_steps": []
        }, llm_config):
        
        if "__end__" not in s:
            print(s)
            print("----")
            if eval(config["SAVE_DIALOGUE"]):
                with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
                    f.write(repr(s) + "\n")
                    f.write("----\n")
            
            # time.sleep(5)
            # Print to console
            log_file.write(f"{s}\n")
            log_file.write("----\n")
            log_file.flush()
    log_file.write(f"=== Session ended at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
    if eval(config["SAVE_DIALOGUE"]):
        with open(f"{WORKING_DIRECTORY}/his.txt", "a") as f:
            f.write(f"=== Session ended at {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n\n")
print("End, check the log file for details")


Setting GPU_AVAILABLE == True
GPU_AVAILABLE: True
##############################################################################################
##  Working directory: /nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools ##
##############################################################################################
Start, check the log file for details
supervisor is processing!!!!!
{'inputs': '\nplease conduct a OER screening study to find out the best system to use as catalyst for OER reaction.\nYou must decide a screening strategy for selecting candidates materials and performing relavent DFT calculation to evaluate the OER activity.\nPlease only consider O only and skip the study of OH and OOH for now. \nAvailable systems can be found in the default dataset. you must use Nsite < 20. Please use VASP as the calculator. \n', 'plan': [], 'past_steps': []}


/home/ziqiw/.conda/envs/agent-ursa/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: WARNING! tool_choice is not default parameter.
                tool_choice was transferred to model_kwargs.
                Please confirm that tool_choice is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


input_tokens: 1759, output_tokens: 14
================================== Ai Message ==================================

[{'id': 'toolu_01DwdSaBmADRMu7LEPLkjTRP', 'input': {}, 'name': 'inspect_my_canvas', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool Calls:
  inspect_my_canvas (toolu_01DwdSaBmADRMu7LEPLkjTRP)
 Call ID: toolu_01DwdSaBmADRMu7LEPLkjTRP
  Args:

================================= Tool Message =================================
Name: inspect_my_canvas

[]

input_tokens: 1823, output_tokens: 34
================================== Ai Message ==================================

[{'id': 'toolu_0185UPhwZdobscTQ5pRRybQn', 'input': {'only_get_updates': False}, 'name': 'inspect_explog', 'type': 'tool_use', 'caller': {'type': 'direct'}}]
Tool Calls:
  inspect_explog (toolu_0185UPhwZdobscTQ5pRRybQn)
 Call ID: toolu_0185UPhwZdobscTQ5pRRybQn
  Args:
    only_get_updates: False

================================= Tool Message =================================
Name: inspect_explog



100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 528971/528971 [01:19<00:00, 6664.49it/s]


Found 29375 stable entries given the stability criteria.
##################### CANVAS #######################
{
'OER_Agent_Expertise_and_Capabilities': '\n## OER Agent Expertise and Capabilities\n\n### What I Can Help With:\n1. **Dataset Screening & Filtering**: Filter and sort materials database based on stability, composition, structure, and other properties\n2. **Candidate Selection**: Identify promising OER catalyst candidates from materials database\n3. **Literature Search**: Conduct arXiv searches for relevant OER research and screening strategies\n4. **DFT Workflow Management**: \n   - Submit bulk relaxation calculations\n   - Submit surface/termination relaxation calculations\n   - Submit adsorption calculations (O and OH)\n5. **Surface Analysis**:\n   - Rank surface terminations based on coordination analysis\n   - Identify and analyze adsorption sites\n6. **Progress Tracking**: Monitor calculation status and results through EXPLOG\n7. **Data Analysis**: Evaluate OER activity 

KeyboardInterrupt: 

In [ ]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools/canvas.pickle", "rb") as f:
    lastCanvas = pickle.load(f)

In [8]:
lastCanvas["OER_stable_entries_df"]

,Composition,MaterialId,Reduced Formula,Elements,NSites,Crystal System,Dimensionality Cheon,Bandgap,Disorder Probability,average_HHI_P,average_HHI_P_excluding_OHCNPS,max_HHI_P,average_HHI_R,average_HHI_R_excluding_OHCNPS,max_HHI_R
298967,Eu1Mn1O6Sb1,8a3e128c32,EuMnSbO6,"[O, Mn, Sb, Eu]",9,trigonal,3D,0.0006,0.78893,2444,6333,9500,1256,2767,3400


In [9]:
var.my_RESOURCE_DIRECTORY

{}

In [7]:
EXPLOG.init(Path(WORKING_DIRECTORY)/"TEMP_vasp_calcs_testtest", "test")

In [8]:
EXPLOG.update_log()

{}

In [7]:
with open("/nfs/turbo/coe-venkvis/ziqiw-turbo/material_agent/OER_test_wAllTools/TEMP_vasp_calcs/explog.pkl", "rb") as f:
    tmpexplog = pickle.load(f)

RecursionError: maximum recursion depth exceeded

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,129caa3e7d,Co-based perovskite-like structure (LaCoSbO6) ...,"NSites=9, trigonal, 3D, Bandgap=1.33 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
1,2730bcba45,Co-based oxide (InCoAsO6) with low disorder pr...,"NSites=9, trigonal, 3D, Bandgap=0.51 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
2,859bfbde9f,Ce-Co oxide (CeCoAsO6) with moderate bandgap (...,"NSites=9, trigonal, 3D, Bandgap=1.17 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
3,8b19878c1f,"Ce-Co phosphate (CeCo2PO8) with two Co atoms, ...","NSites=12, triclinic, 3D, Bandgap=0.15 eV, Dis...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
4,c2e2511a3d,Ba-Ni-Ir oxide (BaNiIrO6) with near-metallic b...,"NSites=9, trigonal, Bandgap=0.003 eV, Disorder...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
5,8668c079e1,Sc-Ni-Bi oxide (ScNi2BiO6) with two Ni atoms a...,"NSites=10, trigonal, 3D, Bandgap=0.82 eV, Diso...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
6,4d1be4578b,"Ni-Ir oxide (NiIr3O8) with three Ir atoms, mon...","NSites=12, monoclinic, 3D, Disorder=0.15",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
7,d9a54f6d10,"Li-Ni oxide (LiNi3O8) with three Ni atoms, nea...","NSites=12, trigonal, Bandgap=0.002 eV, Disorde...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
8,141a2241cc,Sr-Mn-W oxide (SrMnWO6) with trigonal structur...,"NSites=9, trigonal, Bandgap=2.17 eV, Disorder=...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN
9,9272d7a1fe,Nd-Mn-Pt oxide (NdMnPtO6) with near-metallic b...,"NSites=9, trigonal, 3D, Bandgap=0.01 eV, Disor...",<gnome_dreams_oer_screening.oer.oer_study.OER_...,NaN,NaN


In [12]:
id = EXPLOG.add_process("8b19878c1f", "bulk_relaxtion")

In [13]:
id

'0'